# 01 — Paired McNemar Tests, Error Overlap, and Error Migration


> **Post-shared-task analysis.** Gold test labels were public when this analysis was
> designed. Nothing in this notebook changes the official CI=0.035 submission or the
> third-place ranking. Results are retrospective and must not be described as untouched
> test-set estimates.

This notebook compares all ten eligible Task 1b English test submissions. It
computes exact paired McNemar tests with Holm correction, pairwise error-set
overlap/Jaccard, and row-level error migration for predeclared comparisons.

In [ ]:
from pathlib import Path
from collections import Counter
import csv, io, json, os, urllib.request, zipfile

import numpy as np
import pandas as pd


def find_repo_root():
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidate in candidates:
        if (candidate / 'README.md').exists() and (candidate / 'Test').exists():
            return candidate.resolve()
    raise FileNotFoundError('Run this notebook from the IE2026-HalDetect repository.')


ROOT = find_repo_root()
HERE = ROOT / 'post_task_analysis'
CACHE = HERE / 'cache'
OUTPUT = HERE / 'outputs'
CACHE.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)

GOLD_URL = (
    'https://huggingface.co/datasets/QCRI/ImageEval-ArabicNLP26/'
    'resolve/main/task1b/test_en.jsonl'
)
gold_override = os.getenv('IMAGEEVAL_TEST_GOLD')
GOLD_PATH = Path(gold_override) if gold_override else CACHE / 'test_en.jsonl'
if not GOLD_PATH.exists():
    print('Downloading released gold test JSONL...')
    urllib.request.urlretrieve(GOLD_URL, GOLD_PATH)


def read_gold(path=GOLD_PATH):
    rows = [json.loads(line) for line in Path(path).read_text(encoding='utf-8').splitlines()]
    assert len(rows) == 1000, f'Expected 1,000 gold items, found {len(rows)}'
    frame = pd.DataFrame(rows)
    assert frame['id'].is_unique
    assert frame['labels'].map(lambda x: len(x) == 3 and sum(x) == 1).all()
    frame['gold_idx'] = frame['labels'].map(lambda x: x.index(True))
    return frame


def load_prediction_zip(path, gold):
    path = Path(path)
    assert path.exists(), path
    with zipfile.ZipFile(path) as archive:
        csv_names = [name for name in archive.namelist() if name.lower().endswith('.csv')]
        assert len(csv_names) == 1, (path, csv_names)
        with io.TextIOWrapper(archive.open(csv_names[0]), encoding='utf-8-sig') as handle:
            rows = list(csv.DictReader(handle))
    raw = pd.DataFrame(rows)
    required = {'id', 'statement_index', 'prediction'}
    assert required.issubset(raw.columns), (path, raw.columns)
    raw['statement_index'] = raw['statement_index'].astype(int)
    raw['pred_bool'] = raw['prediction'].str.strip().str.lower().map(
        {'true': True, 'false': False})
    assert raw['pred_bool'].notna().all(), f'Unparseable prediction in {path}'
    assert not raw.duplicated(['id', 'statement_index']).any()
    assert set(raw['statement_index']) == {0, 1, 2}
    grouped = raw.sort_values(['id', 'statement_index']).groupby('id', sort=False)
    vectors = grouped['pred_bool'].apply(list)
    assert vectors.map(len).eq(3).all()
    assert set(vectors.index) == set(gold['id']), f'ID mismatch in {path}'

    result = gold[['id', 'gold_idx']].copy()
    by_id = vectors.to_dict()
    result['pred_vector'] = result['id'].map(by_id)
    result['format_valid'] = result['pred_vector'].map(lambda x: sum(x) == 1)
    result['pred_idx'] = result['pred_vector'].map(
        lambda x: x.index(True) if sum(x) == 1 else np.nan)
    result['correct'] = result['format_valid'] & result['pred_idx'].eq(result['gold_idx'])
    result['error'] = ~result['correct']
    return result


SUBMISSIONS = {
    'Elimination': ROOT / 'Test/qwen2p5-3b-7b/All COT variations/cot-elimination/prediction_en.zip',
    'Socratic': ROOT / 'Test/qwen2p5-3b-7b/All COT variations/cot-socratic/prediction_en.zip',
    'Devils advocate': ROOT / 'Test/qwen2p5-3b-7b/All COT variations/cot-devils-advocate/prediction_en.zip',
    'Evidence first': ROOT / 'Test/qwen2p5-3b-7b/All COT variations/cot-evidence-first/prediction_en.zip',
    'Attribute checklist': ROOT / 'Test/qwen2p5-3b-7b/All COT variations/cot-attribute-checklist/prediction_en.zip',
    'Confidence ranked': ROOT / 'Test/qwen2p5-3b-7b/All COT variations/cot-confidence-ranked/prediction_en.zip',
    'QLoRA 2,000': ROOT / 'Test/qwen2p5-3b-7b/qlora-q7b-2k-image/prediction_en.zip',
    'QLoRA 2,348': ROOT / 'Test/qwen2p5-3b-7b/qlora-q7b-2p3k-image/prediction_en.zip',
    'QLoRA 2,600': ROOT / 'Test/qwen2p5-3b-7b/qlora-q7b-2p6k-image/prediction_en.zip',
    'QLoRA 3,000 legacy': ROOT / 'Test/qwen2p5-3b-7b/qlora-q7b-3k-image/prediction_en.zip',
}

gold = read_gold()
predictions = {name: load_prediction_zip(path, gold) for name, path in SUBMISSIONS.items()}
summary = pd.DataFrame([
    {
        'system': name,
        'n': len(frame),
        'errors': int(frame['error'].sum()),
        'CI': frame['error'].mean(),
        'accuracy': frame['correct'].mean(),
        'format_failures': int((~frame['format_valid']).sum()),
    }
    for name, frame in predictions.items()
]).sort_values(['CI', 'system']).reset_index(drop=True)
display(summary)

## Exact paired McNemar tests

In [ ]:
from itertools import combinations
from scipy.stats import binomtest


def holm_adjust(p_values):
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    adjusted = np.empty_like(p_values)
    running = 0.0
    m = len(p_values)
    for rank, index in enumerate(order):
        running = max(running, (m - rank) * p_values[index])
        adjusted[index] = min(1.0, running)
    return adjusted


rows = []
for system_a, system_b in combinations(SUBMISSIONS, 2):
    a = predictions[system_a].set_index('id').loc[gold['id']]
    b = predictions[system_b].set_index('id').loc[gold['id']]
    a_correct = a['correct'].to_numpy()
    b_correct = b['correct'].to_numpy()
    a_only = int((a_correct & ~b_correct).sum())
    b_only = int((~a_correct & b_correct).sum())
    discordant = a_only + b_only
    p_exact = binomtest(a_only, discordant, 0.5).pvalue if discordant else 1.0
    rows.append({
        'system_a': system_a,
        'system_b': system_b,
        'CI_a': (~a_correct).mean(),
        'CI_b': (~b_correct).mean(),
        'delta_CI_a_minus_b': (~a_correct).mean() - (~b_correct).mean(),
        'a_only_correct': a_only,
        'b_only_correct': b_only,
        'discordant': discordant,
        'mcnemar_exact_p': p_exact,
    })
mcnemar = pd.DataFrame(rows)
mcnemar['holm_p'] = holm_adjust(mcnemar['mcnemar_exact_p'])
mcnemar['holm_significant_0.05'] = mcnemar['holm_p'] < 0.05
mcnemar.to_csv(OUTPUT / 'pairwise_mcnemar_exact.csv', index=False)
display(mcnemar.sort_values(['holm_p', 'mcnemar_exact_p']).head(20))

## Error overlap and Jaccard

In [ ]:
error_sets = {
    name: set(frame.loc[frame['error'], 'id']) for name, frame in predictions.items()
}
overlap_rows = []
for system_a, system_b in combinations(SUBMISSIONS, 2):
    a, b = error_sets[system_a], error_sets[system_b]
    overlap_rows.append({
        'system_a': system_a,
        'system_b': system_b,
        'errors_a': len(a),
        'errors_b': len(b),
        'intersection': len(a & b),
        'union': len(a | b),
        'jaccard': len(a & b) / len(a | b) if a | b else 1.0,
        'a_only': len(a - b),
        'b_only': len(b - a),
    })
overlap = pd.DataFrame(overlap_rows)
overlap.to_csv(OUTPUT / 'pairwise_error_overlap.csv', index=False)
display(overlap.sort_values('jaccard').head(20))

import matplotlib.pyplot as plt
import seaborn as sns
names = list(SUBMISSIONS)
matrix = pd.DataFrame(np.eye(len(names)), index=names, columns=names)
for row in overlap.itertuples():
    matrix.loc[row.system_a, row.system_b] = row.jaccard
    matrix.loc[row.system_b, row.system_a] = row.jaccard
plt.figure(figsize=(10, 8))
sns.heatmap(matrix, annot=True, fmt='.2f', cmap='viridis', vmin=0, vmax=1)
plt.title('Jaccard similarity of error sets')
plt.tight_layout()
plt.savefig(OUTPUT / 'error_jaccard_heatmap.png', dpi=180)
plt.show()

## Error migration

In [ ]:
MIGRATION_PAIRS = [
    ('Attribute checklist', 'QLoRA 2,600'),
    ('QLoRA 2,348', 'QLoRA 2,600'),
    ('Elimination', 'QLoRA 2,600'),
]
migration_frames = []
for system_a, system_b in MIGRATION_PAIRS:
    a = predictions[system_a][['id', 'pred_idx', 'correct']].rename(
        columns={'pred_idx': 'pred_a', 'correct': 'correct_a'})
    b = predictions[system_b][['id', 'pred_idx', 'correct']].rename(
        columns={'pred_idx': 'pred_b', 'correct': 'correct_b'})
    merged = gold.merge(a, on='id').merge(b, on='id')
    conditions = [
        merged['correct_a'] & merged['correct_b'],
        merged['correct_a'] & ~merged['correct_b'],
        ~merged['correct_a'] & merged['correct_b'],
    ]
    labels = ['both_correct', 'a_only_correct', 'b_only_correct']
    merged['migration'] = np.select(conditions, labels, default='both_wrong')
    merged['system_a'] = system_a
    merged['system_b'] = system_b
    migration_frames.append(merged)
migration = pd.concat(migration_frames, ignore_index=True)
migration['statements_json'] = migration['statements'].map(json.dumps)
keep = [
    'system_a', 'system_b', 'id', 'country', 'category', 'subcategory',
    'gold_idx', 'pred_a', 'pred_b', 'correct_a', 'correct_b', 'migration',
    'image', 'statements_json',
]
migration[keep].to_csv(OUTPUT / 'error_migration_items.csv', index=False)
migration_summary = migration.groupby(
    ['system_a', 'system_b', 'migration']).size().rename('n').reset_index()
migration_summary.to_csv(OUTPUT / 'error_migration_summary.csv', index=False)
display(migration_summary)